In [ ]:
# setup pyspark environment
import os
import sys

if sys.platform == "darwin":
    os.environ["SPARK_HOME"] = "/Users/hubert/Documents/dev/python/data-analysis-pyspark/spark-4.0.0-bin-hadoop3"
    os.environ["PATH"] += os.pathsep + "/Users/hubert/Documents/dev/python/data-analysis-pyspark/spark-4.0.0-bin-hadoop3/bin"

In [ ]:
%pip install pandas
%pip install pyarrow

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MyApp") \
    .config("spark.driver.extraJavaOptions", "-Dio.netty.tryReflectionSetAccessible=true") \
    .config("spark.executor.extraJavaOptions", "-Dio.netty.tryReflectionSetAccessible=true") \
    .config("spark.sql.execution.arrow.maxRecordsPerBatch", 10000) \
    .getOrCreate()

In [ ]:
import pyspark.sql.functions as F 
import pandas as pd

df = spark.createDataFrame(pd.DataFrame({'hi': ['hello', 'hi', 'hey']}))

df.select(F.upper(F.col('hi'))).show()

In [ ]:
import pyspark.sql.types as T
df2 = spark.createDataFrame(pd.DataFrame({'temps': [32.0, 98.0, 68.0]}), schema=T.StructType([T.StructField("temps", T.DoubleType(), True)]))

@F.pandas_udf(T.DoubleType())
def f_to_c(temps: pd.Series) -> pd.Series:
    return (temps - 32) * 5.0/9.0 

df2.select(f_to_c(F.col('temps')).alias('temps_celsius')).show()

df2.show()

df2.withColumn('temps_celsius', f_to_c(F.col('temps'))).show()

df2.show()

In [ ]:
from time import sleep 
from typing import Iterator 

@F.pandas_udf(T.DoubleType()) 
def f_to_c_batch(temps: Iterator[pd.Series]) -> Iterator[pd.Series]:
    sleep(5)
    for batch in temps:
        yield (batch - 32) * 5.0/9.0


df2.withColumn('temps_celsius', f_to_c_batch(F.col('temps'))).show()

In [ ]:
df3 = spark.createDataFrame(pd.DataFrame({"year": [2020, 2022, 2024], "month": [1, 6, 12], "day": [15, 20, 25]})
                            , schema=T.StructType([T.StructField("year", T.IntegerType(), True),
                                                  T.StructField("month", T.IntegerType(), True),
                                                  T.StructField("day", T.IntegerType(), True)]))
df3.show()

In [ ]:
from typing import Tuple 

@F.pandas_udf(T.DateType())
def create_date(year_mo_da: Iterator[Tuple[pd.Series, pd.Series, pd.Series]]) -> Iterator[pd.Series]:
    for year, month, day in year_mo_da:
        yield pd.to_datetime({'year': year, 'month': month, 'day': day})

df3.withColumn('date', create_date(F.col('year'), F.col('month'), F.col('day'))).show()

In [ ]:
#series udf without decorators

exo_9_1 = pd.Series(["red", "blue", "blue", "yellow"])

def color_to_num(color_series: pd.Series) -> pd.Series:
    return color_series.map({"red": 1, "blue": 2, "yellow": 3})

print(color_to_num(exo_9_1))

color_to_num_udf = F.pandas_udf(color_to_num, returnType=T.IntegerType())

def color_to_num2(color_series: pd.Series) -> pd.Series:
    return color_series.apply(
        lambda c: {"red": 1, "blue": 2, "yellow": 3}.get(c, 0)
    )
color_to_num2_udf = F.pandas_udf(color_to_num2, returnType=T.IntegerType())
print(color_to_num2(exo_9_1))

Grouped Aggregator UDFs

In [ ]:
%pip install scikit-learn
from sklearn.linear_model import LinearRegression

In [ ]:
def rate_of_change_temp(day: pd.Series, temp: pd.Series) -> float:
    model = LinearRegression()
    X = day.astype(int).values.reshape(-1, 1) # convert to integer np array and reshape to a single column for sklearn
    y = temp.astype(float).values
    model.fit(X, y)
    return model.coef_[0]

rate_of_change_temp_udf = F.pandas_udf(rate_of_change_temp, returnType=T.DoubleType())

In [ ]:
test_temps = pd.DataFrame({
    "day": [1, 2, 3, 4, 5],
    "temp": [30.0, 32.0, 34.0, 36.0, 38.0]
})

In [ ]:
print(rate_of_change_temp(test_temps['day'], test_temps['temp']))

In [ ]:
import utils.gsodUtils as gsodUtils
import os

gsod69015093121DataPath = "data/69015093121.csv"
gsod70000126492DataPath = "data/70000126492.csv"

if (not os.path.exists(gsod69015093121DataPath) or
    not os.path.exists(gsod70000126492DataPath)):
    gsodUtils.download_gsod_data("69015093121", 2024, gsod69015093121DataPath)
    gsodUtils.download_gsod_data("70000126492", 2024, gsod70000126492DataPath)



In [ ]:
gsod6 = spark.read.csv(gsod69015093121DataPath, header=True, inferSchema=True)
gsod7 = spark.read.csv(gsod70000126492DataPath, header=True, inferSchema=True)

gsodRaw = gsod6.union(gsod7)
gsodRaw.printSchema()
gsodRaw.show(5)

In [ ]:
import pyspark.sql.functions as F

gsod = gsodRaw.select(
    F.col("DATE"), 
    F.col("STATION").alias("STN"), 
    F.col("TEMP")
    ) \
    .withColumn("year", F.year(F.col("DATE"))) \
    .withColumn("month", F.month(F.col("DATE"))) \
    .withColumn("day", F.day(F.col("DATE")))

gsod.show(2)


In [ ]:
result = gsod.groupBy(F.col("STN"), F.col("year"), F.col("month")) \
    .agg(rate_of_change_temp_udf(gsod["day"], gsod["TEMP"])) \
    .alias("rate_of_change_temp")

result.show()

In [ ]:
def scale_temp(temp_by_day: pd.DataFrame) -> pd.DataFrame:
    """ Returns a simple normalization of the temperature for a site
    If the temperature is constant for the whole window, default to 0.5
    """
    temp = temp_by_day['TEMP']
    answer = temp_by_day[["STN", "year", "month", "day", "TEMP"]]
    if(temp.min() == temp.max()):
        return answer.assign(temp_norm = 0.5)
    return answer.assign(
        temp_norm = (temp - temp.min()) / (temp.max() - temp.min())
    )

In [ ]:
gsod.show(2)
gsod_map = gsod.groupBy(F.col("STN"), F.col("year"), F.col("month")).applyInPandas(
    scale_temp,
    schema=T.StructType([
        T.StructField("STN", T.IntegerType(), True),
        T.StructField("year", T.IntegerType(), True),
        T.StructField("month", T.IntegerType(), True),
        T.StructField("day", T.IntegerType(), True),
        T.StructField("TEMP", T.DoubleType(), True),
        T.StructField("temp_norm", T.DoubleType(), True),
    ])
)
gsod_map.show(5)

In [ ]:
# you can unit test pandas udf functions locally by pulling the underlying function with func and calling it.

gsod_local = gsod.where((F.col("year") == 2024) & (F.col("month") == 1)).toPandas()
gsod_local.head()

print(rate_of_change_temp_udf.func(gsod_local['day'], gsod_local['TEMP']))
